In [71]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [72]:
df = pd.read_csv("uber.csv")

In [73]:
df.head()

,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [74]:
print(df.isnull().sum())

Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64


In [75]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [76]:
df = df[
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]

In [77]:
def haversine(lon1, lat1, lon2, lat2):
    R = 6371
    dlon = np.radians(lon2 - lon1)
    dlat = np.radians(lat2 - lat1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c
df['distance'] = haversine(
    df['pickup_longitude'],
    df['pickup_latitude'],
    df['dropoff_longitude'],
    df['dropoff_latitude']
)

In [78]:
df = df[(df['distance'] > 0) & (df['distance'] < 50)]

In [79]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month

In [80]:
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 100)]


In [81]:
df['fare_amount'] = 50 + (df['distance'] * 15)

In [82]:
df = df.drop(columns=['pickup_datetime', 'key'], errors='ignore')

In [83]:
# correlation
print("\nCorrelation Matrix:\n")
print(df.corr())


Correlation Matrix:

                   Unnamed: 0  fare_amount  pickup_longitude  pickup_latitude  \
Unnamed: 0           1.000000    -0.000687         -0.000420         0.000228   
fare_amount         -0.000687     1.000000          0.005380        -0.002873   
pickup_longitude    -0.000420     0.005380          1.000000        -0.994006   
pickup_latitude      0.000228    -0.002873         -0.994006         1.000000   
dropoff_longitude   -0.000368     0.004498          0.999885        -0.993995   
dropoff_latitude     0.000215    -0.002290         -0.993975         0.999929   
passenger_count      0.002827     0.008213          0.009175        -0.009306   
distance            -0.000687     1.000000          0.005380        -0.002873   
hour                 0.000166    -0.032439          0.001836        -0.000986   
day                  0.000470     0.000486          0.019517        -0.020069   
month                0.001953     0.012250         -0.007490         0.008013   

     

In [84]:
# feature and target values
X = df[['pickup_longitude', 'pickup_latitude',
        'dropoff_longitude', 'dropoff_latitude',
        'passenger_count', 'distance',
        'hour', 'day', 'month']]

y = df['fare_amount']
print("FINAL FEATURES:", X.columns) 

FINAL FEATURES: Index(['pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'passenger_count', 'distance', 'hour', 'day',
       'month'],
      dtype='object')


In [85]:
# train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [86]:
# model
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [87]:
# Random Forest
rf = RandomForestRegressor(
    n_estimators=20,     
    max_depth=10,        
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=20, n_jobs=-1, random_state=42)

In [88]:
# evaluation 
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)
print("\nLinear Regression RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("Linear Regression R2:", r2_score(y_test, lr_pred))
print("\nRandom Forest RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("Random Forest R2:", r2_score(y_test, rf_pred))



Linear Regression RMSE: 1.088814226320608e-13
Linear Regression R2: 1.0

Random Forest RMSE: 0.06712906689178139
Random Forest R2: 0.9999983935343902


In [89]:
# for pickle file
if r2_score(y_test, rf_pred) > r2_score(y_test, lr_pred):
    best_model = rf
    print("\nRandom Forest selected")
else:
    best_model = lr
    print("\nLinear Regression selected")

pickle.dump(best_model, open("model.pkl", "wb"))

print("\nmodel.pkl saved successfully!")


Linear Regression selected

model.pkl saved successfully!
